# 🐎 馬の走り 全身バイオメカ解析（Colab・ViTPose+）

真横スローの走行動画から、**関節角度・重心・走法クロック・円弧モデル・かき込み度・振り幅・
軌跡の幅・リズム・負荷(推定)** を出し、**オーバーレイ画像**と**「この馬の位置」レポート**を作ります。

## ⚠️ まず GPU を有効化
「ランタイム」→「ランタイムのタイプを変更」→ **GPU** → 保存

## 使い方
上から順に ▶。①セットアップは初回数分。②でツールを書き出し、③で動画、④で姿勢推定、⑤解析、⑥レポート。

> 正直メモ: 姿勢推定(ViTPose+/mmpose)はバージョンに敏感です。④が失敗したら、エラーに合わせて
> `mmcv`/`mmpose` のバージョンを調整してください。重心・力・負荷は**モデル推定（実測ではない）**。

## ① セットアップ（初回のみ・数分）

In [ ]:
# mmpose(ViTPose+) 一式。バージョンは動けばOK、ダメなら調整。
!pip -q install -U openmim yt-dlp opencv-python-headless cairosvg imageio imageio-ffmpeg > /dev/null
!mim install -q "mmengine>=0.7" "mmcv>=2.0.0,<2.2.0" "mmdet>=3.0.0" "mmpose>=1.1.0" > /dev/null
import torch
print(("✅ GPU: "+torch.cuda.get_device_name(0)) if torch.cuda.is_available()
      else "❌ GPU無効 → ランタイムのタイプでGPUを選び①から再実行")

## ② ツールを書き出す（一瞬）

In [ ]:
%%writefile biomech.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""全身バイオメカ解析エンジン（馬・MVP）  biomech.py

骨格キーポイント（DeepLabCut SuperAnimal-Quadruped → paddock_gait.extract_target_horse）
から、下記を算出する。pandas/cv2非依存のコア（frames = 各フレーム {bodypart:(x,y)} のリスト）。

【実測（＝キーポイントから計算。確実）】
  ・関節角度: 頸・肩まわり・腕節(前膝)・膝(後)・飛節
  ・馬の重心（第12〜13肋骨相当。既存cog_pointと同式）
  ・走法クロック位置: 蹄が重心を原点にどの時計位置か
       12時=浮き / 3時=推進 / 6時=接地・回転 / 9時=制動
  ・歩幅/体長比・毎秒◯完歩（回転数）・後肢踏み込み位置（体長比）・接地/空中%（暫定）

【推定（＝実測ではない。モデル仮定）】
  ・各部の負荷(1〜5)・力の向き … 2D単眼からは真の力は測れないためモデル推定。
    参考動画も「負荷メーター（モデル推定・実測ではない）」。同方式の相対ヒート。

使い方:
  import biomech; m = biomech.analyze(frames, fps=120)   # framesは単一トラック
  biomech.analyze_h5("race_uma05....h5", fps=120)         # h5から直接
  python3 biomech.py --selftest
"""
import math
import statistics as st

PCUTOFF = 0.6

# 角度を測る関節定義（左サイド優先・無ければ右）  name: (A, 中心, C)
JOINTS = {
    "頸":   ("back_base",       "neck_base",       "nose"),
    "肩":   ("back_base",       "{F}_thai",        "{F}_knee"),
    "腕節": ("{F}_thai",        "{F}_knee",        "{F}_paw"),
    "膝":   ("back_end",        "{B}_thai",        "{B}_knee"),
    "飛節": ("{B}_thai",        "{B}_knee",        "{B}_paw"),
}


# ───────── 幾何 ─────────
def _ang(a, b, c):
    """点Bでの ∠ABC（0〜180度）。"""
    bax, bay = a[0] - b[0], a[1] - b[1]
    bcx, bcy = c[0] - b[0], c[1] - b[1]
    na = math.hypot(bax, bay) or 1e-9
    nc = math.hypot(bcx, bcy) or 1e-9
    cos = max(-1.0, min(1.0, (bax * bcx + bay * bcy) / (na * nc)))
    return math.degrees(math.acos(cos))


def _dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def _get(fr, name):
    return fr.get(name)


def _side_points(fr, tmpl, side_f, side_b):
    """テンプレ {F}/{B} を左右に差し替えて3点を取得。"""
    out = []
    for p in tmpl:
        p = p.replace("{F}", side_f).replace("{B}", side_b)
        out.append(_get(fr, p))
    return out


# ───────── 重心 ─────────
def cog(fr):
    front = _get(fr, "back_base") or _get(fr, "neck_base")
    rear = _get(fr, "tail_base") or _get(fr, "back_end") or _get(fr, "back_middle")
    if not (front and rear):
        return None
    vx, vy = rear[0] - front[0], rear[1] - front[1]
    L = math.hypot(vx, vy) or 1.0
    fx, fy = front[0] + 0.52 * vx, front[1] + 0.52 * vy
    px, py = -vy, vx
    if py < 0:
        px, py = vy, -vx
    depth = 0.22 * L
    return (fx + depth * px / L, fy + depth * py / L)


def body_len(fr):
    n, t = _get(fr, "neck_base"), _get(fr, "tail_base")
    return _dist(n, t) if (n and t) else None


def travel_dir(fr):
    """進行方向の水平単位（頭が前）。neck_base→noseのx符号。"""
    n, nose = _get(fr, "neck_base"), _get(fr, "nose") or _get(fr, "neck_end")
    t = _get(fr, "tail_base")
    if n and nose:
        dx = nose[0] - n[0]
    elif n and t:
        dx = n[0] - t[0]
    else:
        dx = 1.0
    return 1.0 if dx >= 0 else -1.0


# ───────── 関節角度 ─────────
def joint_angles(fr):
    out = {}
    for name, tmpl in JOINTS.items():
        got = None
        for sf, sb in (("front_left", "back_left"), ("front_right", "back_right")):
            pts = _side_points(fr, tmpl, sf, sb)
            if all(pts):
                got = _ang(*pts)
                break
        if got is not None:
            out[name] = round(got, 1)
    return out


# ───────── 走法クロック ─────────
def clock_of(hoof, c, tdir):
    """蹄vが重心cを原点に、どの時計位置か。
    12時=真上(浮き) 3時=進行方向(推進) 6時=真下(接地) 9時=後方(制動)。
    戻り: (hour[0-12), ラベル)。"""
    vx, vy = (hoof[0] - c[0]) * tdir, hoof[1] - c[1]   # x: 前方+になるよう進行方向を掛ける
    up = -vy            # 画面上方向が+
    fwd = vx            # 前方が+
    # 12時(上)から時計回りに測る角度
    phi = math.degrees(math.atan2(fwd, up)) % 360.0
    hour = (phi / 30.0) % 12.0
    if up >= abs(fwd):
        lab = "浮き(12時)"
    elif fwd >= abs(up) and fwd > 0:
        lab = "推進(3時)"
    elif -vy <= -abs(fwd):
        lab = "接地・回転(6時)"
    else:
        lab = "制動(9時)"
    return hour, lab


def gait_clock(fr):
    """接地側の蹄（最下）で走法位置を代表。前後それぞれも返す。"""
    c = cog(fr)
    if not c:
        return None
    tdir = travel_dir(fr)
    res = {}
    hooves = {}
    for key, bps in (("前", ("front_left_paw", "front_right_paw")),
                     ("後", ("back_left_paw", "back_right_paw"))):
        p = _get(fr, bps[0]) or _get(fr, bps[1])
        if p:
            hooves[key] = p
            res[key] = clock_of(p, c, tdir)
    if not hooves:
        return None
    # 代表＝最も下にある蹄（接地に近い）
    low = max(hooves.items(), key=lambda kv: kv[1][1])
    res["代表"] = clock_of(low[1], c, tdir)
    return res


# ───────── 歩幅・回転・接地 ─────────
def stride_metrics(frames, fps):
    paws, bodylens, lows = [], [], []
    for fr in frames:
        p = _get(fr, "back_left_paw") or _get(fr, "back_right_paw") \
            or _get(fr, "front_left_paw") or _get(fr, "front_right_paw")
        if p:
            paws.append(p[0])
            lows.append(p[1])
        bl = body_len(fr)
        if bl:
            bodylens.append(bl)
    out = {}
    if len(bodylens) >= 1:
        out["体長px"] = round(st.median(bodylens), 1)
    # 完歩/秒: 蹄x座標のゼロ交差(前後往復)の頻度
    if len(paws) > 4 and fps:
        m = st.mean(paws)
        sign = [1 if x >= m else -1 for x in paws]
        crossings = sum(1 for i in range(1, len(sign)) if sign[i] != sign[i - 1])
        strides = crossings / 2.0
        secs = len(paws) / fps
        if secs > 0:
            out["完歩_毎秒"] = round(strides / secs, 2)
    # 歩幅/体長比: 蹄xの可動幅 / 体長
    if len(paws) > 2 and bodylens:
        amp = max(paws) - min(paws)
        out["歩幅_体長比"] = round(amp / st.median(bodylens), 2)
    # 接地%/空中%(暫定): 最下蹄yの上側ばらつきで近似
    if len(lows) > 4:
        g = max(lows)                       # 最も低い=地面に近い
        contact = sum(1 for y in lows if y >= g - 0.03 * (st.median(bodylens) if bodylens else 100))
        out["接地率_暫定"] = round(100 * contact / len(lows), 0)
    return out


# ───────── 体フレーム・蹄の軌跡（手順3: 胴の長さを物差しに） ─────────
def body_frame(fr):
    """重心原点・体長1の座標系 (c, f前方unit, u背側unit, L)。無ければNone。"""
    c = cog(fr)
    n, t = _get(fr, "neck_base"), _get(fr, "tail_base")
    if not (c and n and t):
        return None
    fx, fy = n[0] - t[0], n[1] - t[1]            # 前方(尾→首)
    L = math.hypot(fx, fy) or 1.0
    f = (fx / L, fy / L)
    u = (-f[1], f[0])                            # 垂直
    b = _get(fr, "belly_bottom")
    if b and ((b[0] - c[0]) * u[0] + (b[1] - c[1]) * u[1]) > 0:
        u = (-u[0], -u[1])                       # uを背側(腹と反対)に向ける
    return c, f, u, L


def to_body(p, bf):
    c, f, u, L = bf
    dx, dy = p[0] - c[0], p[1] - c[1]
    return ((dx * f[0] + dy * f[1]) / L, (dx * u[0] + dy * u[1]) / L)  # (前方, 背側)


def hoof_track(frames, bp):
    out = []
    for fr in frames:
        p, bf = _get(fr, bp), body_frame(fr)
        if p and bf:
            out.append(to_body(p, bf))
    return out


def _first_track(frames, bps):
    for bp in bps:
        t = hoof_track(frames, bp)
        if len(t) >= 4:
            return t
    return []


def trajectory_width(frames, bp_pair):
    """蹄の通り道（円弧モデル）の前後幅＝「一歩の大きさ」。体長比で返す
    （参考: 前肢1.19・後肢1.32）。併せて横÷縦の平たさも返す。"""
    t = _first_track(frames, bp_pair)
    if len(t) < 4:
        return None
    w = max(p[0] for p in t) - min(p[0] for p in t)   # 前後幅（体長比）
    h = max(p[1] for p in t) - min(p[1] for p in t)   # 上下幅（体長比）
    return {"幅_体長比": round(w, 2), "横÷縦": round(w / h, 2) if h > 1e-6 else None}


def swing_amplitude(frames, side_pair=("front_left", "front_right")):
    """前肢の振り幅: 脚ベクトル(thai→paw)の鉛直からの前後角。前へ/後ろへ/振り幅(deg)。"""
    angs = []
    for fr in frames:
        bf = body_frame(fr)
        thai = paw = None
        for s in side_pair:
            thai = thai or _get(fr, f"{s}_thai")
            paw = paw or _get(fr, f"{s}_paw")
        if bf and thai and paw:
            c, f, u, L = bf
            vx, vy = paw[0] - thai[0], paw[1] - thai[1]
            fwd = vx * f[0] + vy * f[1]
            down = -(vx * u[0] + vy * u[1])             # 脚は下へ垂れる=正
            angs.append(math.degrees(math.atan2(fwd, down)))  # 0=真下 +前 -後
    if len(angs) < 4:
        return None
    return {"前へ": round(max(angs)), "後ろへ": round(-min(angs)),
            "振り幅": round(max(angs) - min(angs))}


def dig_in_degree(frames, bp_pair):
    """かき込み度 0〜1。着地(最下点)が前方最大reachよりどれだけ後ろか。
    1=蹄を後ろへ引きながら着く(かき込み型)、0=前で突く(ステップ型)。"""
    t = _first_track(frames, bp_pair)
    if len(t) < 6:
        return None
    contact = min(t, key=lambda p: p[1])           # 最も背側マイナス=最下(接地)
    fwd = [p[0] for p in t]
    rng = max(fwd) - min(fwd)
    if rng < 1e-6:
        return None
    return round(max(0.0, min(1.0, (max(fwd) - contact[0]) / rng)), 2)


def rhythm_overlap(frames):
    """リズムの重なり 0〜1。蹄の前後位置信号を1完歩ずらした自己相関のピーク。
    1=完全に周期的(安定)。"""
    sig = []
    for fr in frames:
        bf = body_frame(fr)
        p = _get(fr, "back_left_paw") or _get(fr, "back_right_paw") \
            or _get(fr, "front_left_paw") or _get(fr, "front_right_paw")
        if bf and p:
            sig.append(to_body(p, bf)[0])
    if len(sig) < 8:
        return None
    m = sum(sig) / len(sig)
    s = [x - m for x in sig]
    den = sum(x * x for x in s) or 1e-9
    best = 0.0
    for lag in range(2, len(s) // 2):
        v = sum(s[i] * s[i - lag] for i in range(lag, len(s))) / den
        best = max(best, v)
    return round(best, 2)


def hind_reach(frames):
    """後肢の踏み込み: 後蹄がいちばん前へ出た位置を 臀端0〜肩端1 で返す
    （参考: 0.66）。"""
    vals = []
    for fr in frames:
        rear = _get(fr, "back_end") or _get(fr, "tail_base")       # 臀端
        front = (_get(fr, "front_left_thai") or _get(fr, "front_right_thai")
                 or _get(fr, "neck_base"))                         # 肩端
        paw = _get(fr, "back_left_paw") or _get(fr, "back_right_paw")
        if rear and front and paw:
            ax = (front[0] - rear[0], front[1] - rear[1])
            L2 = ax[0] ** 2 + ax[1] ** 2 or 1e-9
            t = ((paw[0] - rear[0]) * ax[0] + (paw[1] - rear[1]) * ax[1]) / L2
            vals.append(t)
    return round(max(vals), 2) if vals else None


# ───────── 負荷推定（モデル・実測ではない） ─────────
LOAD_SEGMENTS = ["肩・胸", "肘・上腕", "腕節・前腕", "屈腱・繋靭帯", "球節・繋", "蹄"]


def load_estimate(fr):
    """各部の相対負荷(1〜5)の“推定”。接地・体重を受ける側ほど、遠位ほど高い。
    ※2D単眼からの力は測れない。角度と接地状況からのヒューリスティック。"""
    c = cog(fr)
    if not c:
        return {}
    tdir = travel_dir(fr)
    # 接地前肢（最下の前蹄）を支持肢とみなす
    cand = [p for p in (_get(fr, "front_left_paw"), _get(fr, "front_right_paw")) if p]
    if not cand:
        return {}
    paw = max(cand, key=lambda p: p[1])
    hour, lab = clock_of(paw, c, tdir)
    # 支持性: 蹄が重心の真下〜後方(6〜9時)にあるほど体重を受ける＝高負荷
    weightbearing = 1.0 if paw[1] > c[1] else 0.4      # 蹄が重心より下
    brake = 1.0 if (paw[0] - c[0]) * tdir < 0 else 0.7  # 重心より後方=制動寄り
    base = 5.0 * weightbearing * brake
    # 遠位(蹄・球節)を重く、近位(肩)を軽めに配分
    weights = [0.55, 0.6, 0.7, 0.9, 1.0, 0.95]
    out = {}
    for seg, w in zip(LOAD_SEGMENTS, weights):
        out[seg] = max(1, min(5, round(base * w)))
    return out


# ───────── まとめ ─────────
def analyze(frames, fps=None):
    frames = [f for f in frames if f]
    if not frames:
        return {"ok": False, "reason": "有効フレーム無し"}
    mid = frames[len(frames) // 2]
    res = {
        "ok": True, "n_frames": len(frames),
        "angles": joint_angles(mid),
        "clock": gait_clock(mid),
        "load_est": load_estimate(mid),
        "stride": stride_metrics(frames, fps) if fps else {},
    }
    c = cog(mid)
    res["cog"] = (round(c[0], 1), round(c[1], 1)) if c else None
    # 走法の総合位置(代表)を全フレーム平均で安定化
    hours = []
    for fr in frames:
        gc = gait_clock(fr)
        if gc and "代表" in gc:
            hours.append(gc["代表"][0])
    if hours:
        # 円周平均
        s = sum(math.sin(math.radians(h * 30)) for h in hours)
        co = sum(math.cos(math.radians(h * 30)) for h in hours)
        mh = (math.degrees(math.atan2(s, co)) / 30.0) % 12.0
        res["走法位置_時"] = round(mh, 1)
        res["走法傾向"] = ("推進寄り" if 1.5 <= mh < 4.5 else
                           "接地・回転" if 4.5 <= mh < 7.5 else
                           "制動寄り" if 7.5 <= mh < 10.5 else "浮き寄り")
    # 追加メトリクス（複数フレーム）
    res["かき込み度"] = dig_in_degree(frames, ("front_left_paw", "front_right_paw"))
    res["振り幅_前肢"] = swing_amplitude(frames, ("front_left", "front_right"))
    res["振り幅_後肢"] = swing_amplitude(frames, ("back_left", "back_right"))
    res["リズム重なり"] = rhythm_overlap(frames)
    res["軌跡幅_前肢"] = trajectory_width(frames, ("front_left_paw", "front_right_paw"))
    res["軌跡幅_後肢"] = trajectory_width(frames, ("back_left_paw", "back_right_paw"))
    res["踏み込み_臀端0肩端1"] = hind_reach(frames)
    res["関節ROM"] = joint_rom(frames)
    return res


def joint_rom(frames):
    """各関節が1完歩で動く幅（最大角−最小角, deg）。飛節・腕節の可動幅など。"""
    series = {}
    for fr in frames:
        for k, v in joint_angles(fr).items():
            series.setdefault(k, []).append(v)
    return {k: round(max(v) - min(v), 1) for k, v in series.items() if len(v) >= 2}


def frames_from_h5(h5_path, pcutoff=PCUTOFF):
    import pandas as pd
    import paddock_gait
    df = paddock_gait.extract_target_horse(pd.read_hdf(h5_path))
    bps = list(df.columns.get_level_values(0).unique())
    frames = []
    for _, row in df.iterrows():
        fr = {}
        for bp in bps:
            if (bp, "likelihood") in df.columns and row[(bp, "likelihood")] >= pcutoff:
                x, y = row[(bp, "x")], row[(bp, "y")]
                if x == x and y == y:
                    fr[bp] = (float(x), float(y))
        frames.append(fr)
    return frames


def analyze_h5(h5_path, fps=None, pcutoff=PCUTOFF):
    return analyze(frames_from_h5(h5_path, pcutoff), fps=fps)


def print_report(res):
    if not res.get("ok"):
        print("解析不可:", res.get("reason")); return
    print(f"■ バイオメカ解析（{res['n_frames']}フレーム）")
    print("  関節角度:", "  ".join(f"{k}{v}°" for k, v in res["angles"].items()))
    if res.get("走法位置_時") is not None:
        print(f"  走法クロック: {res['走法位置_時']}時 → {res['走法傾向']}"
              "（12浮き/3推進/6回転/9制動）")
    sw = res.get("振り幅_前肢")
    if sw:
        print(f"  前肢の振り幅: {sw['振り幅']}°（前へ{sw['前へ']}°/後ろへ{sw['後ろへ']}°）")
    if res.get("かき込み度") is not None:
        print(f"  かき込み度: {res['かき込み度']}（1=後ろへ引いて着く/0=前で突く）")
    tf = res.get("軌跡幅_前肢"); tb = res.get("軌跡幅_後肢")
    if tf and tb:
        print(f"  蹄の通り道の幅(体長比): 前肢{tf['幅_体長比']}・後肢{tb['幅_体長比']}")
    if res.get("リズム重なり") is not None:
        print(f"  リズムの重なり: {res['リズム重なり']}（1=周期が安定）")
    if res.get("踏み込み_臀端0肩端1") is not None:
        print(f"  後肢の踏み込み: {res['踏み込み_臀端0肩端1']}（臀端0〜肩端1）")
    if res.get("stride"):
        print("  歩様:", "  ".join(f"{k}={v}" for k, v in res["stride"].items()))
    if res.get("load_est"):
        print("  負荷(推定1-5):", " ".join(f"{k}:{v}" for k, v in res["load_est"].items()))


# ───────── selftest ─────────
def _synthetic(dig_delta=2.2, n=36, period=12):
    """脚が振動する合成フレーム。左向き(頭=左)前進。
    dig_delta: 前肢の接地位相。大→後ろへ引いて着く(かき込み型), 小→前で突く。"""
    frames = []
    Ar, Lf = 60.0, 70.0      # 前肢 前後reach / 持ち上げ
    Ab, Lb = 55.0, 64.0      # 後肢
    for i in range(n):
        p = 2 * math.pi * i / period
        x0 = 600 - i * 2
        neck = (x0 - 130, 200); tail = (x0 + 130, 205); nose = (x0 - 205, 205)
        fr = {"neck_base": neck, "tail_base": tail, "nose": nose,
              "back_base": (x0 - 90, 198), "back_end": (x0 + 95, 205),
              "back_middle": (x0, 196), "belly_bottom": (x0, 272)}
        # 前肢(左): reach=前後, lift=接地(dig_delta)で0
        reach = Ar * math.cos(p)
        lift = Lf * (0.5 - 0.5 * math.cos(p - dig_delta))
        ft = (x0 - 70, 235)
        fp = (ft[0] - reach, 360 - lift)
        fr["front_left_thai"] = ft
        fr["front_left_paw"] = fp
        fr["front_left_knee"] = ((ft[0] + fp[0]) / 2 - 12, (ft[1] + fp[1]) / 2)
        # 後肢(左): 逆位相
        reachb = Ab * math.cos(p + math.pi)
        liftb = Lb * (0.5 - 0.5 * math.cos(p + math.pi - 0.6))
        bt = (x0 + 70, 235)
        bp_ = (bt[0] - reachb, 360 - liftb)
        fr["back_left_thai"] = bt
        fr["back_left_paw"] = bp_
        fr["back_left_knee"] = ((bt[0] + bp_[0]) / 2 + 12, (bt[1] + bp_[1]) / 2)
        frames.append(fr)
    return frames


def selftest():
    rake = analyze(_synthetic(dig_delta=2.2), fps=120)   # 後ろへ引いて着く
    stab = analyze(_synthetic(dig_delta=0.6), fps=120)   # 前で突く
    print_report(rake); print()
    assert rake["ok"] and stab["ok"]
    for need in ("頸", "腕節", "飛節", "膝"):
        assert need in rake["angles"], f"{need}角度が無い"
    assert rake.get("走法位置_時") is not None, "走法クロックが出ない"
    # かき込み度: rake(後ろ引き) > stab(前突き)
    assert rake["かき込み度"] is not None and stab["かき込み度"] is not None
    assert rake["かき込み度"] > stab["かき込み度"], \
        f"かき込み度の向きが逆: rake{rake['かき込み度']} vs stab{stab['かき込み度']}"
    assert rake["かき込み度"] >= 0.5 and stab["かき込み度"] <= 0.4, "かき込み度の域が不適切"
    # 振り幅・軌跡幅・リズム・踏み込みが算出される
    assert rake["振り幅_前肢"] and rake["振り幅_前肢"]["振り幅"] > 0
    assert rake["軌跡幅_前肢"]["幅_体長比"] > 0
    assert rake["リズム重なり"] is not None
    assert rake["踏み込み_臀端0肩端1"] is not None
    # 負荷推定 1-5・遠位>=近位
    le = rake["load_est"]
    assert le and all(1 <= v <= 5 for v in le.values())
    assert le["球節・繋"] >= le["肩・胸"], "遠位ほど高負荷のモデルが崩れている"
    print("SELFTEST PASSED（角度・走法クロック・かき込み度・振り幅・軌跡幅・"
          "リズム・踏み込み・負荷推定が算出でき、かき込み型と突き型を判別）")


def main():
    import argparse
    ap = argparse.ArgumentParser(description="全身バイオメカ解析（馬MVP）")
    ap.add_argument("--h5", help="DeepLabCut h5")
    ap.add_argument("--fps", type=float)
    ap.add_argument("--selftest", action="store_true")
    a = ap.parse_args()
    if a.selftest:
        selftest(); return
    if not a.h5:
        ap.error("--h5 か --selftest を指定")
    print_report(analyze_h5(a.h5, fps=a.fps))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile pose_adapter.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""ViTPose+（AP-10K・馬17点）→ biomech キーポイント名 への変換  pose_adapter.py

参考資料は ViTPose+（AP-10K 17点）で姿勢推定している。biomech.py は
DeepLabCut系の名前（neck_base, back_base, front_left_thai …）で書かれているので、
両者を橋渡しするマッピング層。中間の背点(back_base/back_middle)や腹点(belly_bottom)は
17点に無いので、首〜尾と四肢から幾何的に合成する。

AP-10K 17点の標準並び（index: 名前）:
  0 左目 1 右目 2 鼻 3 首 4 尾根 5 左肩 6 左肘 7 左前蹄
  8 右肩 9 右肘 10 右前蹄 11 左股 12 左膝 13 左後蹄 14 右股 15 右膝 16 右後蹄

使い方:
  frame = ap10k_to_biomech(kps)      # kps: [(x,y,conf), ...] len17
  frames = [ap10k_to_biomech(k) for k in per_frame_kps]
  res = biomech.analyze(frames, fps=...)
"""
AP10K = ["left_eye", "right_eye", "nose", "neck", "root_of_tail",
         "left_shoulder", "left_elbow", "left_front_paw",
         "right_shoulder", "right_elbow", "right_front_paw",
         "left_hip", "left_knee", "left_back_paw",
         "right_hip", "right_knee", "right_back_paw"]
IDX = {n: i for i, n in enumerate(AP10K)}

# AP-10K名 → biomech名（直接対応ぶん）
DIRECT = {
    "nose": "nose", "neck": "neck_base", "root_of_tail": "tail_base",
    "left_shoulder": "front_left_thai", "left_elbow": "front_left_knee",
    "left_front_paw": "front_left_paw",
    "right_shoulder": "front_right_thai", "right_elbow": "front_right_knee",
    "right_front_paw": "front_right_paw",
    "left_hip": "back_left_thai", "left_knee": "back_left_knee",
    "left_back_paw": "back_left_paw",
    "right_hip": "back_right_thai", "right_knee": "back_right_knee",
    "right_back_paw": "back_right_paw",
}


def _lerp(a, b, t):
    return (a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t)


def ap10k_to_biomech(kps, pcutoff=0.3):
    """kps: 長さ17の [(x,y,conf), ...]（または [(x,y), ...]）。
    戻り: biomechが使う {name:(x,y)} のフレーム辞書。信頼度不足は除外。"""
    def g(name):
        i = IDX[name]
        if i >= len(kps):
            return None
        p = kps[i]
        if len(p) >= 3 and p[2] < pcutoff:
            return None
        x, y = p[0], p[1]
        if x != x or y != y:
            return None
        return (float(x), float(y))

    fr = {}
    for apn, bmn in DIRECT.items():
        p = g(apn)
        if p:
            fr[bmn] = p

    neck, tail = fr.get("neck_base"), fr.get("tail_base")
    if neck and tail:
        # 背線上の中間点を合成（17点に無い）
        fr.setdefault("back_base", _lerp(neck, tail, 0.15))
        fr["back_middle"] = _lerp(neck, tail, 0.5)
        fr.setdefault("back_end", _lerp(neck, tail, 0.85))
        # 腹点 belly_bottom: 背線中点から、蹄側(腹方向)へ体長の約28%下ろす
        hooves = [fr[k] for k in ("front_left_paw", "front_right_paw",
                                  "back_left_paw", "back_right_paw") if k in fr]
        bm = fr["back_middle"]
        if hooves:
            mh = (sum(h[0] for h in hooves) / len(hooves),
                  sum(h[1] for h in hooves) / len(hooves))
            L = ((tail[0] - neck[0]) ** 2 + (tail[1] - neck[1]) ** 2) ** 0.5 or 1.0
            vx, vy = mh[0] - bm[0], mh[1] - bm[1]
            vlen = (vx * vx + vy * vy) ** 0.5 or 1.0
            fr["belly_bottom"] = (bm[0] + 0.28 * L * vx / vlen,
                                  bm[1] + 0.28 * L * vy / vlen)
    return fr


def selftest():
    import math
    import biomech
    # 合成の17点フレーム列（左向き・頭=左、前肢が振動）を作って変換→解析
    frames = []
    for i in range(30):
        p = 2 * math.pi * i / 12
        x0 = 600 - i * 2
        reach = 60 * math.cos(p)
        lift = 70 * (0.5 - 0.5 * math.cos(p - 2.2))
        kps = [None] * 17
        def put(n, xy): kps[IDX[n]] = (xy[0], xy[1], 0.9)
        put("nose", (x0 - 205, 205)); put("neck", (x0 - 130, 200))
        put("root_of_tail", (x0 + 130, 205))
        put("left_shoulder", (x0 - 70, 235)); put("left_elbow", (x0 - 80, 300))
        put("left_front_paw", (x0 - 70 - reach, 360 - lift))
        put("right_shoulder", (x0 - 66, 238)); put("right_elbow", (x0 - 76, 302))
        put("right_front_paw", (x0 - 66 - reach, 362 - lift))
        put("left_hip", (x0 + 70, 235)); put("left_knee", (x0 + 80, 300))
        put("left_back_paw", (x0 + 70 - 55 * math.cos(p + math.pi), 360))
        put("right_hip", (x0 + 66, 238)); put("right_knee", (x0 + 76, 302))
        put("right_back_paw", (x0 + 66 - 55 * math.cos(p + math.pi), 362))
        frames.append(ap10k_to_biomech(kps))
    # 合成した背点・腹点が入っているか
    f0 = frames[0]
    for need in ("neck_base", "tail_base", "back_base", "back_middle",
                 "back_end", "belly_bottom", "front_left_thai", "back_left_knee"):
        assert need in f0, f"変換で {need} が欠落"
    # biomechが解析でき、主要メトリクスが出る
    res = biomech.analyze(frames, fps=120)
    assert res["ok"], "変換フレームをbiomechが解析できない"
    assert res["angles"] and res.get("走法位置_時") is not None
    assert res["かき込み度"] is not None and res["振り幅_前肢"]
    print("変換例:", {k: (round(v[0]), round(v[1])) for k, v in list(f0.items())[:5]})
    print("解析OK: 走法", res["走法位置_時"], "時 / かき込み", res["かき込み度"],
          "/ 前肢振り幅", res["振り幅_前肢"]["振り幅"], "°")
    print("SELFTEST PASSED（AP-10K 17点→biomech名の変換、背点/腹点の合成、"
          "biomech解析まで通る）")


if __name__ == "__main__":
    import sys
    if "--selftest" in sys.argv:
        selftest()
    else:
        print("使い方: python3 pose_adapter.py --selftest")


In [ ]:
%%writefile biomech_overlay.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""バイオメカ解析オーバーレイ（手順4の見た目）  biomech_overlay.py

biomech.py の計測を、1フレームの馬の上に重ねて描く。参考画面の中心ビジュアル:
  ・骨格線（負荷1〜5で色分け: 青→赤）
  ・関節角度ラベル（頸・肩・腕節・膝・飛節）
  ・馬の重心（黄）＋重心を通る鉛直線
  ・走法クロック（12浮き/3推進/6回転/9制動＋現在位置の赤◆）
  ・円弧モデル（重心を原点にした蹄の通り道）インセット
  ・力の向き矢印・負荷の凡例・まとめ

実写では image_b64（真横フレーム）に重ねる。無ければ暗背景に骨格だけ描く。
SVGを書き出し、chromiumでPNG化できる（--png）。

使い方:
  python3 biomech_overlay.py --selftest          # 合成データで描画→HTML/PNG
  # 実パイプラインでは biomech.frames_from_h5 → render() に frames と image を渡す
"""
import math

HEAT = ["#2f6fe0", "#17a2b8", "#38b000", "#f59f00", "#e03131"]  # 負荷1→5

# 描く骨格線（利用可能なキーポイントのみ）
EDGES_SPINE = [("neck_base", "back_base"), ("back_base", "back_middle"),
               ("back_middle", "back_end"), ("back_end", "tail_base"),
               ("neck_base", "nose")]
# 肢: (親, 子, 負荷セグメント名)
EDGES_LEG = {
    "front_left": [("front_left_thai", "front_left_knee", "腕節・前腕"),
                   ("front_left_knee", "front_left_paw", "球節・繋")],
    "back_left": [("back_left_thai", "back_left_knee", "腕節・前腕"),
                  ("back_left_knee", "back_left_paw", "球節・繋")],
}
# 角度を出す関節: 表示名 → 中心キーポイント
ANGLE_AT = {"頸": "neck_base", "腕節": "front_left_knee", "飛節": "back_left_knee"}


def _heat(load):
    return HEAT[max(1, min(5, int(round(load or 1)))) - 1]


def render(frames, idx, analysis, image_b64=None, W=1000, H=620):
    import biomech
    fr = frames[idx]

    def P(bp):
        return fr.get(bp)
    s = [f'<svg viewBox="0 0 {W} {H}" width="100%" font-family="sans-serif">']
    if image_b64:
        s.append(f'<image href="data:image/webp;base64,{image_b64}" x="0" y="0" '
                 f'width="{W}" height="{H}" preserveAspectRatio="xMidYMid slice"/>')
        s.append(f'<rect width="{W}" height="{H}" fill="#000" opacity="0.18"/>')
    else:
        s.append(f'<rect width="{W}" height="{H}" fill="#0d0d0c"/>')

    load = analysis.get("load_est") or {}
    # 背骨（白太）
    for a, b in EDGES_SPINE:
        pa, pb = P(a), P(b)
        if pa and pb:
            s.append(f'<line x1="{pa[0]:.0f}" y1="{pa[1]:.0f}" x2="{pb[0]:.0f}" '
                     f'y2="{pb[1]:.0f}" stroke="#eee" stroke-width="3" opacity="0.8"/>')
    # 肢（負荷色）
    for leg, edges in EDGES_LEG.items():
        for a, b, seg in edges:
            pa, pb = P(a), P(b)
            if pa and pb:
                col = _heat(load.get(seg, load.get("球節・繋", 3)))
                s.append(f'<line x1="{pa[0]:.0f}" y1="{pa[1]:.0f}" x2="{pb[0]:.0f}" '
                         f'y2="{pb[1]:.0f}" stroke="{col}" stroke-width="7" '
                         f'stroke-linecap="round"/>')
    # 関節点＋角度ラベル
    ang = analysis.get("angles", {})
    for name, bp in ANGLE_AT.items():
        p = P(bp)
        if p and name in ang:
            s.append(f'<circle cx="{p[0]:.0f}" cy="{p[1]:.0f}" r="5" fill="#fff"/>')
            s.append(f'<rect x="{p[0]+8:.0f}" y="{p[1]-20:.0f}" width="78" height="20" '
                     f'rx="3" fill="#000b"/>'
                     f'<text x="{p[0]+12:.0f}" y="{p[1]-5:.0f}" font-size="13" '
                     f'fill="#ffd43b">{name} {ang[name]:.0f}°</text>')
    # 重心＋鉛直線
    c = biomech.cog(fr)
    if c:
        s.append(f'<line x1="{c[0]:.0f}" y1="20" x2="{c[0]:.0f}" y2="{H-120}" '
                 f'stroke="#ffd43b" stroke-width="1.5" stroke-dasharray="6 5" opacity="0.7"/>')
        s.append(f'<circle cx="{c[0]:.0f}" cy="{c[1]:.0f}" r="9" fill="#ffd43b" '
                 f'stroke="#000" stroke-width="2"/>')
        s.append(f'<text x="{c[0]+12:.0f}" y="{c[1]+4:.0f}" font-size="12" fill="#ffd43b">馬の重心</text>')
        # 力の向き矢印（走法傾向で前/後ろ）
        tend = analysis.get("走法傾向", "")
        dirx = -60 if "推進" in tend else 60 if "制動" in tend else 0
        if dirx:
            s.append(f'<line x1="{c[0]:.0f}" y1="{c[1]:.0f}" x2="{c[0]+dirx:.0f}" '
                     f'y2="{c[1]-40:.0f}" stroke="#fff" stroke-width="4"/>')
            lab = "推進(前)" if dirx < 0 else "制動(後ろ)"
            s.append(f'<text x="{c[0]+dirx-20:.0f}" y="{c[1]-46:.0f}" font-size="12" '
                     f'fill="#fff">力の向き {lab}</text>')

    # 走法クロック（右上）
    _clock_inset(s, analysis, W - 150, 90, 66)
    # 円弧モデル（左下）
    _arc_inset(s, frames, 150, H - 90, 120)
    # 負荷凡例＋まとめ（下帯）
    _legend(s, analysis, W, H)
    s.append("</svg>")
    return "".join(s)


def _clock_inset(s, analysis, cx, cy, r):
    s.append(f'<circle cx="{cx}" cy="{cy}" r="{r}" fill="#000a" stroke="#555" stroke-width="1.5"/>')
    for hour, lab in ((12, "浮"), (3, "推"), (6, "回"), (9, "制")):
        a = math.radians((hour % 12) * 30 - 90)
        x, y = cx + (r - 12) * math.cos(a), cy + (r - 12) * math.sin(a)
        s.append(f'<text x="{x:.0f}" y="{y+4:.0f}" text-anchor="middle" font-size="11" '
                 f'fill="#aaa">{lab}</text>')
    h = analysis.get("走法位置_時")
    if h is not None:
        a = math.radians((h % 12) * 30 - 90)
        x, y = cx + (r - 28) * math.cos(a), cy + (r - 28) * math.sin(a)
        s.append(f'<path d="M{x:.0f},{y-7:.0f} L{x+7:.0f},{y:.0f} L{x:.0f},{y+7:.0f} '
                 f'L{x-7:.0f},{y:.0f} Z" fill="#e03131"/>')
        s.append(f'<text x="{cx}" y="{cy+r+16:.0f}" text-anchor="middle" font-size="12" '
                 f'fill="#fff">走法 {h}時 {analysis.get("走法傾向","")}</text>')


def _arc_inset(s, frames, cx, cy, size):
    import biomech
    s.append(f'<rect x="{cx-size:.0f}" y="{cy-size*0.6:.0f}" width="{size*2}" '
             f'height="{size*1.2:.0f}" rx="6" fill="#000a" stroke="#555"/>')
    s.append(f'<text x="{cx-size+6:.0f}" y="{cy-size*0.6+14:.0f}" font-size="10" '
             f'fill="#bbb">円弧モデル（重心が原点）</text>')
    for bps, col in ((("front_left_paw",), "#ffd43b"), (("back_left_paw",), "#4dabf7")):
        t = biomech.hoof_track(frames, bps[0])
        if len(t) < 3:
            continue
        pts = []
        for fwd, up in t:
            # 体長比→インセット座標（前方=右, 背側上=上）
            px = cx + fwd * size * 0.9
            py = cy - up * size * 0.9
            pts.append(f"{px:.0f},{py:.0f}")
        s.append(f'<polyline points="{" ".join(pts)}" fill="none" stroke="{col}" '
                 f'stroke-width="2" opacity="0.9"/>')
    s.append(f'<circle cx="{cx:.0f}" cy="{cy:.0f}" r="3" fill="#fff"/>')


def _legend(s, analysis, W, H):
    y = H - 44
    s.append(f'<rect x="0" y="{y-6}" width="{W}" height="50" fill="#000c"/>')
    for i, c in enumerate(HEAT):
        s.append(f'<rect x="{14+i*26}" y="{y}" width="22" height="12" fill="{c}"/>')
        s.append(f'<text x="{20+i*26}" y="{y+22:.0f}" font-size="9" fill="#ccc">{i+1}</text>')
    s.append(f'<text x="14" y="{y-10:.0f}" font-size="10" fill="#aaa">負荷(推定1→5)</text>')
    # まとめ
    sw = analysis.get("振り幅_前肢") or {}
    tf = (analysis.get("軌跡幅_前肢") or {}).get("幅_体長比")
    parts = []
    if sw:
        parts.append(f"前肢振り{sw.get('振り幅')}°")
    if analysis.get("かき込み度") is not None:
        parts.append(f"かき込み{analysis['かき込み度']}")
    if tf is not None:
        parts.append(f"軌跡幅{tf}")
    if analysis.get("リズム重なり") is not None:
        parts.append(f"リズム{analysis['リズム重なり']}")
    s.append(f'<text x="170" y="{y+8:.0f}" font-size="12" fill="#fff">'
             f'まとめ: {" / ".join(parts)}</text>')
    s.append(f'<text x="170" y="{y+24:.0f}" font-size="9.5" fill="#999">'
             f'※重心・力・負荷はモデル推定（実測ではない）</text>')


def to_png(svg_or_html_path, png_path, W=1000, H=620):
    import glob, os, subprocess
    ch = (glob.glob("/opt/pw-browsers/chromium-*/chrome-linux*/chrome") or [None])[0]
    if not ch:
        return None
    subprocess.run([ch, "--headless", "--no-sandbox", "--disable-gpu",
                    "--hide-scrollbars", "--force-device-scale-factor=2",
                    f"--window-size={W},{H}", f"--screenshot={png_path}",
                    f"file://{svg_or_html_path}"], capture_output=True)
    return png_path if os.path.exists(png_path) else None


def selftest():
    import biomech, tempfile, os
    frames = biomech._synthetic(dig_delta=2.2)
    analysis = biomech.analyze(frames, fps=120)
    svg = render(frames, len(frames) // 2, analysis, W=1000, H=620)
    assert svg.startswith("<svg") and "走法" in svg and "馬の重心" in svg
    assert "円弧モデル" in svg and "負荷(推定" in svg
    d = tempfile.mkdtemp()
    html = os.path.join(d, "ov.html")
    open(html, "w", encoding="utf-8").write(
        f'<!doctype html><meta charset="utf-8"><body style="margin:0">{svg}</body>')
    print(f"HTML: {html}")
    print("SELFTEST PASSED（オーバーレイSVG生成: 骨格・角度・重心・走法クロック・"
          "円弧モデル・負荷凡例・まとめを含む）")
    return html


def main():
    import argparse
    ap = argparse.ArgumentParser(description="バイオメカ解析オーバーレイ")
    ap.add_argument("--selftest", action="store_true")
    ap.add_argument("--png", help="PNG出力先")
    a = ap.parse_args()
    if a.selftest:
        html = selftest()
        if a.png:
            print("PNG:", to_png(html, a.png))
        return
    ap.error("--selftest を使うか、パイプラインから render() を呼んでください")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile biomech_report.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""バイオメカ報告「この馬の位置」比較チャート  biomech_report.py

biomech.analyze() の結果から、参考スライド『この馬の位置を図で示す』と同じ
横軸ドットプロットのレポートHTMLを生成する。
  ・赤いひし形＝この馬の値
  ・灰色の点＝これまでに解析した馬の分布（0頭なら位置づけは未確定）
  ・各行に根拠 A/B/C（A=成績と直接比べた研究, B=間接, C=成績との研究なし）
  ・並び順＝競走成績への重要順（上ほど根拠が強い）
  ・「良い/悪い」の判定ではなく“位置”を見る図。区切り(軸範囲)は暫定。

出典: 体節質量比 Buchner 1997 / de Leva 1996、歩幅・回転と成績 Morrice-West 2021 /
      Barrey 2001 / Schrurs 2022、円弧モデル・かき込み型 小林寛道（東大名誉教授）。

使い方:
  python3 biomech_report.py --h5 race_uma05...h5 --fps 120 --name メイショウアキカゼ -o report.html
  python3 biomech_report.py --selftest
"""
import argparse
import html

# 行定義: (ラベル, 根拠, 軸lo, 軸hi, 単位, res→値 の取り出し)
#   並び順＝競走成績への重要順。軸範囲は暫定。
ROWS = [
    ("歩幅（胴の長さ比）", "A", 3.69, 4.29, "倍",
     lambda r: (r.get("stride") or {}).get("歩幅_体長比")),
    ("リズムの重なり", "A", 0.93, 1.05, "",
     lambda r: r.get("リズム重なり")),
    ("軌跡の幅（胴の長さ比）", "B", 1.16, 1.34, "",
     lambda r: (r.get("軌跡幅_前肢") or {}).get("幅_体長比")),
    ("腰の振り幅", "B", 65, 79, "°",
     lambda r: (r.get("振り幅_後肢") or {}).get("振り幅")),
    ("前肢の振り幅", "B", 69, 83, "°",
     lambda r: (r.get("振り幅_前肢") or {}).get("振り幅")),
    ("人の重心のずれ（前が＋）", "B", 3, 15, "%",
     lambda r: r.get("人重心ずれ")),            # phase2（人体ポーズ）→ None
    ("飛節の動く幅", "C", 57, 71, "°",
     lambda r: (r.get("関節ROM") or {}).get("飛節")),
    ("腕節の動く幅", "C", 82, 96, "°",
     lambda r: (r.get("関節ROM") or {}).get("腕節")),
]

GRADE_COLOR = {"A": "#2f9e44", "B": "#e8842a", "C": "#868e96"}


def _fmt(v, unit):
    if v is None:
        return "—"
    s = f"{v:+.0f}" if unit == "%" else (f"{v:.2f}" if abs(v) < 10 else f"{v:.0f}")
    return s + unit


def _axis_svg(v, lo, hi, priors):
    """1行ぶんの横軸SVG。赤ひし形＝この馬、灰点＝これまでの馬。"""
    W, H = 460, 44
    x0, x1, y = 70, W - 70, 22
    out = [f'<svg viewBox="0 0 {W} {H}" width="{W}" height="{H}" '
           f'font-family="sans-serif">']
    out.append(f'<line x1="{x0}" y1="{y}" x2="{x1}" y2="{y}" stroke="#ced4da" stroke-width="2"/>')
    out.append(f'<text x="{x0-6}" y="{y+4}" text-anchor="end" font-size="10" fill="#868e96">{lo:g}</text>')
    out.append(f'<text x="{x1+6}" y="{y+4}" font-size="10" fill="#868e96">{hi:g}</text>')

    def X(val):
        t = 0.0 if hi == lo else (val - lo) / (hi - lo)
        return x0 + max(0.0, min(1.0, t)) * (x1 - x0)
    for p in (priors or []):
        out.append(f'<circle cx="{X(p):.1f}" cy="{y}" r="4" fill="#adb5bd" opacity="0.7"/>')
    if v is not None:
        cx = X(v)
        out.append(f'<path d="M{cx:.1f},{y-8} L{cx+8:.1f},{y} L{cx:.1f},{y+8} '
                   f'L{cx-8:.1f},{y} Z" fill="#e03131" stroke="#fff" stroke-width="1.5"/>')
    else:
        out.append(f'<text x="{(x0+x1)/2:.0f}" y="{y+4}" text-anchor="middle" '
                   f'font-size="10" fill="#adb5bd">測定なし</text>')
    out.append("</svg>")
    return "".join(out)


def render(res, name="この馬", date="", gait="駈歩・襲歩", priors=None):
    priors = priors or {}
    n_prior = max((len(v) for v in priors.values()), default=0)
    rows_html = []
    for label, grade, lo, hi, unit, getter in ROWS:
        v = getter(res) if res else None
        gc = GRADE_COLOR[grade]
        rows_html.append(
            '<div class="row">'
            f'<div class="lab"><div class="nm">{html.escape(label)}</div>'
            f'<div class="gr" style="color:{gc}">根拠{grade}'
            f'{"：成績と直接比べた研究あり" if grade=="A" else "：間接の研究" if grade=="B" else "：成績との研究なし"}</div></div>'
            f'<div class="ax">{_axis_svg(v, lo, hi, priors.get(label))}</div>'
            f'<div class="val" style="color:{"#e03131" if v is not None else "#adb5bd"}">{_fmt(v, unit)}</div>'
            '</div>')
    undet = ("<b>比べた馬はまだ0頭です。</b>位置の良し悪しはまだ言えません。"
             if n_prior < 3 else f"比べた馬 {n_prior}頭。")
    return f"""<!doctype html><html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>{html.escape(name)} この馬の位置</title><style>
 body{{margin:0;background:#f4f5f7;color:#1b2733;font-family:system-ui,
   -apple-system,"Hiragino Kaku Gothic ProN",sans-serif}}
 .wrap{{max-width:980px;margin:0 auto;background:#fff}}
 .hd{{background:#17345c;color:#fff;padding:14px 20px;display:flex;
   justify-content:space-between;align-items:baseline}}
 .hd .t{{font-size:17px;font-weight:700}} .hd .m{{font-size:12px;opacity:.85}}
 .leg{{display:flex;gap:16px;padding:8px 20px;font-size:11px;color:#495057;
   border-bottom:1px solid #e9ecef;flex-wrap:wrap}}
 .leg b{{color:#e03131}}
 .row{{display:grid;grid-template-columns:230px 460px 1fr;align-items:center;
   padding:4px 20px;border-bottom:1px solid #f1f3f5}}
 .row:nth-child(even){{background:#fafbfc}}
 .lab .nm{{font-size:14px;font-weight:600}} .lab .gr{{font-size:10.5px}}
 .val{{font-size:17px;font-weight:700;text-align:right}}
 .note{{padding:12px 20px;font-size:11.5px;color:#495057;line-height:1.7}}
 .note .warn{{color:#e03131;font-weight:600}}
 .src{{padding:4px 20px 18px;font-size:10.5px;color:#868e96;line-height:1.7}}
</style></head><body><div class="wrap">
<div class="hd"><div class="t">{html.escape(name)}　比較：この馬の位置（同じ歩法の馬と、研究の値）</div>
<div class="m">{html.escape(date)}　{html.escape(gait)}</div></div>
<div class="leg"><span>◆ <b>この馬</b></span>
<span>● これまでに解析した馬（{n_prior}頭）</span>
<span>並び順＝競走成績への重要順（上ほど根拠が強い）</span></div>
{"".join(rows_html)}
<div class="note"><span class="warn">{undet}</span>
　同じ条件の馬が増えるほど、灰色の点で分布が出て位置づけが確かになります。
これは“位置”を見る図で、良い・悪いの判定ではありません。関節角度は測り方が研究と
違うため、研究の値は重ねていません。区切り（軸範囲）は暫定です。</div>
<div class="src">根拠 A＝成績と直接比べた研究あり／B＝間接の研究／C＝成績との研究なし。
研究の値（毎秒の完歩数）：レース2.36±0.12回（日本921頭・Takahashi 2025）、
秒速9.5〜12.5mで2.01〜2.16回（Parsons 2008・Ferrari 2009）。
出典：体節質量比 Buchner 1997／de Leva 1996、歩幅・回転と成績 Morrice-West 2021／
Barrey 2001／Schrurs 2022、円弧モデル・かき込み型 小林寛道（東京大学名誉教授）。
重心・力・負荷はモデル推定で実測ではありません。</div>
</div></body></html>"""


def build(res, out, **kw):
    doc = render(res, **kw)
    with open(out, "w", encoding="utf-8") as f:
        f.write(doc)
    return out


def selftest():
    import biomech
    res = biomech.analyze(biomech._synthetic(dig_delta=2.2), fps=120)
    out = "/tmp/biomech_report_selftest.html"
    build(res, out, name="合成テスト号", date="2026-11-14")
    doc = open(out, encoding="utf-8").read()
    for need in ("この馬の位置", "歩幅（胴の長さ比）", "根拠A", "比べた馬はまだ0頭",
                 "e03131", "飛節の動く幅", "小林寛道"):
        assert need in doc, f"レポートに {need} が無い"
    # 行数 = ROWS 数
    assert doc.count('class="row"') == len(ROWS), "行数が合わない"
    print(f"保存: {out}")
    print("SELFTEST PASSED（この馬の位置レポートを生成・8行・赤マーカー・根拠・"
          "0頭注記・出典を含む）")


def main():
    ap = argparse.ArgumentParser(description="バイオメカ報告『この馬の位置』")
    ap.add_argument("--h5")
    ap.add_argument("--fps", type=float)
    ap.add_argument("--name", default="この馬")
    ap.add_argument("--date", default="")
    ap.add_argument("--gait", default="駈歩・襲歩")
    ap.add_argument("-o", "--out", default="biomech_report.html")
    ap.add_argument("--selftest", action="store_true")
    a = ap.parse_args()
    if a.selftest:
        selftest(); return
    if not a.h5:
        ap.error("--h5 か --selftest を指定")
    import biomech
    res = biomech.analyze_h5(a.h5, fps=a.fps)
    print("保存:", build(res, a.out, name=a.name, date=a.date, gait=a.gait))


if __name__ == "__main__":
    main()


In [ ]:
# 自己検証（計測ロジックが健全か）
!python3 biomech.py --selftest && python3 pose_adapter.py --selftest && python3 biomech_report.py --selftest

## ③ 動画を用意（真横・大きく映る区間）
URLを入れて区間指定でDL（家庭回線/ColabはYouTube可の場合が多い）。
うまくいかない時は、下のアップロードを使う。

In [ ]:
URL   = "https://youtu.be/XXXXXXXX"   # ← 真横の走行が映る動画
START = "0:07"      # 解析したい開始（真横で大きく映る所）
END   = "0:27"      # 終了（10〜20秒でOK）
import subprocess, glob, os
os.makedirs("in", exist_ok=True)
cmd=["yt-dlp","-f","mp4/bestvideo+bestaudio/best","--merge-output-format","mp4",
     "-o","in/clip.%(ext)s"]
if START and END: cmd+=["--download-sections",f"*{START}-{END}","--force-keyframes-at-cuts"]
cmd+=[URL]
try:
    subprocess.run(cmd,check=True); VIDEO=sorted(glob.glob("in/clip.*"))[0]
    print("取得:",VIDEO)
except Exception as e:
    print("DL失敗:",e,"\n→ 下のセルでアップロードしてください"); VIDEO=None

In [ ]:
# 方法B: 手元の真横動画をアップロード（③が失敗した時だけ）
from google.colab import files
up=files.upload()
if up:
    import os; os.makedirs("in",exist_ok=True)
    VIDEO="in/"+list(up.keys())[0]; open(VIDEO,"wb").write(list(up.values())[0])
    print("アップロード:",VIDEO)

## ④ 姿勢推定（ViTPose+ 馬17点）
各コマで馬を検出し、17点を取って**目的馬（最大・手前）**を1頭選びます。
重複コマは自動で間引き、真横に近いコマだけ使います。

In [ ]:
import cv2, numpy as np, math
from mmpose.apis import MMPoseInferencer

# ViTPose+ をAP-10K(動物17点)で。detは既定の動物検出を使う。
infer = MMPoseInferencer(pose2d="td-hm_ViTPose-base_8xb64-210e_ap10k-256x192",
                         det_model="rtmdet-m", det_cat_ids=[0])  # 0=動物(ap10kは1クラス)

cap=cv2.VideoCapture(VIDEO); FPS=cap.get(cv2.CAP_PROP_FPS) or 30.0
raw_kps=[]; frames_bgr=[]
ok,prev=cap.read()
while ok:
    frames_bgr.append(prev); ok,prev=cap.read()
cap.release()
print(f"{len(frames_bgr)}コマ・{FPS:.1f}fps")

def pick_target(preds):
    # 最大バウンディング(手前/大きい)の個体を選ぶ
    best=None
    for p in preds:
        kp=np.array(p["keypoints"]); sc=np.array(p.get("keypoint_scores",[1]*len(kp)))
        w=kp[:,0].max()-kp[:,0].min(); h=kp[:,1].max()-kp[:,1].min()
        area=w*h
        if best is None or area>best[0]:
            best=(area,kp,sc)
    if best is None: return None
    _,kp,sc=best
    return [(float(kp[i,0]),float(kp[i,1]),float(sc[i])) for i in range(len(kp))]

for res in infer(frames_bgr, batch_size=4, show=False):
    preds=res["predictions"][0]
    raw_kps.append(pick_target(preds) if preds else None)
print("推定した有効コマ:",sum(1 for k in raw_kps if k))

## ⑤ 解析（マッピング→計測）

In [ ]:
import pose_adapter, biomech
frames=[pose_adapter.ap10k_to_biomech(k) if k else {} for k in raw_kps]
res=biomech.analyze(frames, fps=FPS)
biomech.print_report(res)

## ⑥ オーバーレイ画像＋「この馬の位置」レポート

In [ ]:
import base64, cairosvg, biomech_overlay, biomech_report
from IPython.display import HTML, Image, display

# 代表コマ（中央付近の有効コマ）にオーバーレイ
idx=max((i for i,f in enumerate(frames) if f.get("neck_base")),
        key=lambda i:abs(i-len(frames)//2), default=len(frames)//2)
H,W=frames_bgr[idx].shape[:2]
ok,buf=cv2.imencode(".webp", frames_bgr[idx]); b64=base64.b64encode(buf).decode()
svg=biomech_overlay.render(frames, idx, res, image_b64=b64, W=W, H=H)
cairosvg.svg2png(bytestring=svg.encode(), write_to="overlay.png", output_width=W*2)
display(Image("overlay.png"))

# レポート
biomech_report.build(res, "report.html", name="この馬", date="", gait="駈歩・襲歩")
display(HTML(open("report.html",encoding="utf-8").read()))

## ⑦ 結果をダウンロード

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("biomech_result","zip",".",)  # overlay.png, report.html 等
for f in ("overlay.png","report.html"):
    try: files.download(f)
    except Exception as e: print(e)